# NAA200R Data Processing
This notebook processes NAA200R.txt file and converts it to NAA200R.csv format

In [8]:
import pandas as pd
import datetime as dt
import re

print("NAA200R Data Processing Started...")

NAA200R Data Processing Started...


In [9]:
# Read the NAA200R.txt file
input_file = 'data/NAA200R.txt'
output_file = 'data/NAA200R_processed.csv'

with open(input_file, 'r') as f:
    lines = f.readlines()

print(f"Read {len(lines)} lines from {input_file}")
print("First few lines:")
for i, line in enumerate(lines[:10]):
    print(f"{i+1:2d}: {line.strip()}")

Read 5930 lines from data/NAA200R.txt
First few lines:
 1: $NAA200R, Daily
 2: Day       Date       Open       High        Low      Close      Volume
 3: === ========== ========== ========== ========== ========== ===========
 4: Mon 07-21-2025     43.903     44.672     43.280     43.593           0
 5: Fri 07-18-2025     44.384     44.470     42.792     42.935           0
 6: Thu 07-17-2025     42.087     44.055     42.087     43.427           0
 7: Wed 07-16-2025     41.664     41.892     39.755     41.693           0
 8: Tue 07-15-2025     43.310     43.367     40.752     40.781           0
 9: Mon 07-14-2025     42.291     43.041     42.291     42.984           0
10: Fri 07-11-2025     43.554     43.554     42.291     42.462           0


In [ ]:
# Process the data
processed_data = []

# Skip header lines (first 3 lines)
data_lines = lines[3:]

for line in data_lines:
    line = line.strip()
    if not line:  # Skip empty lines
        continue

    # Split by whitespace and parse
    parts = line.split()
    if len(parts) >= 7:  # Make sure we have all required fields
        day = parts[0]  # Mon, Tue, etc.
        date_str = parts[1]  # MM-DD-YYYY format
        open_price = float(parts[2])
        high_price = float(parts[3])
        low_price = float(parts[4])
        close_price = float(parts[5])
        volume = int(parts[6])

        # Convert date format from MM-DD-YYYY to M/D/YYYY
        try:
            date_obj = dt.datetime.strptime(date_str, '%m-%d-%Y')
            formatted_date = date_obj.strftime('%-m/%-d/%Y')  # Remove leading zeros
        except ValueError:
            print(f"Warning: Could not parse date {date_str}")
            continue

        processed_data.append({
            'Day': day,
            'Date': formatted_date,
            'Open': open_price,
            'High': high_price,
            'Low': low_price,
            'Close': close_price,
            'Volume': volume
        })

print(f"Processed {len(processed_data)} data records")
print("\nFirst few processed records:")
for i, record in enumerate(processed_data[:5]):
    print(f"{i+1}: {record}")

Processed 5927 data records

First few processed records:
1: {'Day': 'Mon', 'Date': '7/21/2025', 'Open': 43.903, 'High': 44.672, 'Low': 43.28, 'Close': 43.593, 'Volume': 0}
2: {'Day': 'Fri', 'Date': '7/18/2025', 'Open': 44.384, 'High': 44.47, 'Low': 42.792, 'Close': 42.935, 'Volume': 0}
3: {'Day': 'Thu', 'Date': '7/17/2025', 'Open': 42.087, 'High': 44.055, 'Low': 42.087, 'Close': 43.427, 'Volume': 0}
4: {'Day': 'Wed', 'Date': '7/16/2025', 'Open': 41.664, 'High': 41.892, 'Low': 39.755, 'Close': 41.693, 'Volume': 0}
5: {'Day': 'Tue', 'Date': '7/15/2025', 'Open': 43.31, 'High': 43.367, 'Low': 40.752, 'Close': 40.781, 'Volume': 0}


In [11]:
# Create DataFrame and sort by date (newest first)
df = pd.DataFrame(processed_data)

# Convert Date column to datetime for proper sorting
df['Date_obj'] = pd.to_datetime(df['Date'])

# Sort by date with newest first (descending order)
df = df.sort_values('Date_obj', ascending=False)

# Remove the temporary column
df = df.drop('Date_obj', axis=1)

# Reset index to ensure clean numbering
df = df.reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
print("\nDataFrame info:")
print(df.info())
print("\nFirst few rows (should be newest dates):")
print(df.head())
print("\nLast few rows (should be oldest dates):")
print(df.tail())

# Verify date ordering
df_temp = df.copy()
df_temp['Date_obj'] = pd.to_datetime(df_temp['Date'])
print(f"\nDate range verification:")
print(f"First date: {df_temp['Date_obj'].iloc[0]}")
print(f"Last date: {df_temp['Date_obj'].iloc[-1]}")
print(f"Is properly sorted (newest to oldest): {df_temp['Date_obj'].is_monotonic_decreasing}")

DataFrame shape: (5927, 7)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5927 entries, 0 to 5926
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Day     5927 non-null   object 
 1   Date    5927 non-null   object 
 2   Open    5927 non-null   float64
 3   High    5927 non-null   float64
 4   Low     5927 non-null   float64
 5   Close   5927 non-null   float64
 6   Volume  5927 non-null   int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 324.3+ KB
None

First few rows (should be newest dates):
   Day       Date    Open    High     Low   Close  Volume
0  Mon  7/21/2025  43.903  44.672  43.280  43.593       0
1  Fri  7/18/2025  44.384  44.470  42.792  42.935       0
2  Thu  7/17/2025  42.087  44.055  42.087  43.427       0
3  Wed  7/16/2025  41.664  41.892  39.755  41.693       0
4  Tue  7/15/2025  43.310  43.367  40.752  40.781       0

Last few rows (should be oldest dates):
      Day        

In [12]:
# Save to CSV
df.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

# Verify the output
print("\nVerifying output file:")
test_df = pd.read_csv(output_file)
print(f"Output file has {len(test_df)} rows")

print("\nFirst few rows of output (should be newest dates):")
print(test_df.head())

print("\nLast few rows of output (should be oldest dates):")
print(test_df.tail())

# Double-check the date ordering in the saved file
test_df['Date_obj'] = pd.to_datetime(test_df['Date'])
print(f"\nSaved file date verification:")
print(f"First date in file: {test_df['Date_obj'].iloc[0]}")
print(f"Last date in file: {test_df['Date_obj'].iloc[-1]}")
print(f"File is properly sorted (newest to oldest): {test_df['Date_obj'].is_monotonic_decreasing}")

Data saved to data/NAA200R_processed.csv

Verifying output file:
Output file has 5927 rows

First few rows of output (should be newest dates):
   Day       Date    Open    High     Low   Close  Volume
0  Mon  7/21/2025  43.903  44.672  43.280  43.593       0
1  Fri  7/18/2025  44.384  44.470  42.792  42.935       0
2  Thu  7/17/2025  42.087  44.055  42.087  43.427       0
3  Wed  7/16/2025  41.664  41.892  39.755  41.693       0
4  Tue  7/15/2025  43.310  43.367  40.752  40.781       0

Last few rows of output (should be oldest dates):
      Day        Date    Open    High     Low   Close  Volume
5922  Thu    1/3/2002  54.946  54.946  54.946  54.946       0
5923  Wed    1/2/2002  50.969  50.969  50.969  50.969       0
5924  Mon  12/31/2001  50.672  50.672  50.672  50.672       0
5925  Fri  12/28/2001  51.550  51.550  51.550  51.550       0
5926  Thu  12/27/2001  51.331  51.331  51.331  51.331       0

Saved file date verification:
First date in file: 2025-07-21 00:00:00
Last date in fi

In [ ]:
# Optional: Compare with existing NAA200R.csv if it exists
try:
    existing_df = pd.read_csv('data/NAA200R.csv')
    print(f"\nExisting NAA200R.csv has {len(existing_df)} rows")
    print(f"New processed file has {len(test_df)} rows")
    print(f"Difference: {len(test_df) - len(existing_df)} rows")

    print("\nExisting file date range:")
    existing_df['Date_obj'] = pd.to_datetime(existing_df['Date'])
    print(f"From {existing_df['Date_obj'].min()} to {existing_df['Date_obj'].max()}")

    print("\nNew file date range:")
    test_df['Date_obj'] = pd.to_datetime(test_df['Date'])
    print(f"From {test_df['Date_obj'].min()} to {test_df['Date_obj'].max()}")

except FileNotFoundError:
    print("\nNo existing NAA200R.csv found - this is the first processed file")


Existing NAA200R.csv has 5927 rows
New processed file has 5927 rows
Difference: 0 rows

Existing file date range:
From 2001-12-27 00:00:00 to 2025-07-21 00:00:00

New file date range:
From 2001-12-27 00:00:00 to 2025-07-21 00:00:00


In [15]:
# Optional: Replace the original NAA200R.csv with the new processed data

import shutil
shutil.copy2(output_file, 'data/NAA200R.csv')
print("Original NAA200R.csv has been replaced with the processed data")


print("\n✓ NAA200R data processing completed successfully!")

Original NAA200R.csv has been replaced with the processed data

✓ NAA200R data processing completed successfully!
